# Homework 2 - Regression

Dataset: car_fuel_efficiency_2026

In [1]:
import numpy as np
import pandas as pd

In [2]:
url = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'
df = pd.read_csv(url)

columns = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year', 'fuel_efficiency_mpg']
df = df[columns]
df.head()

,engine_displacement,horsepower,vehicle_weight,model_year,fuel_efficiency_mpg
0,2180,243.0,3870,2006,31.9
1,2390,272.0,4210,2008,31.3
2,2320,267.0,4240,1996,27.5
3,2130,258.0,4490,1989,28.5
4,2580,304.0,4510,1994,31.0


## EDA

Distribuição de `fuel_efficiency_mpg`: ela tem cauda longa?

In [3]:
df.fuel_efficiency_mpg.describe()

count    10000.000000
mean        29.997700
std          2.930245
min         19.800000
25%         28.000000
50%         30.000000
75%         31.900000
max         41.200000
Name: fuel_efficiency_mpg, dtype: float64

## Question 1 - Missing values

In [4]:
df.isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

## Question 2 - Median for horsepower

In [5]:
df.horsepower.median()

np.float64(254.0)

## Funções auxiliares (código da aula)

In [6]:
base = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']

def split_data(df, seed):
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
    df_val = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
    df_test = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)
    return df_train, df_val, df_test

def prepare_X(df, fill_value):
    return df[base].fillna(fill_value).values

def train_linear_regression(X, y, r=0.0):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])
    XTX = X.T.dot(X) + r * np.eye(X.shape[1])
    w_full = np.linalg.inv(XTX).dot(X.T).dot(y)
    return w_full[0], w_full[1:]

def rmse(y, y_pred):
    return np.sqrt(((y - y_pred) ** 2).mean())

## Question 3 - Filling NAs

In [7]:
df_train, df_val, df_test = split_data(df, 42)
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values

mean_hp = df_train.horsepower.mean()

for name, fill in [('With 0', 0), ('With mean', mean_hp)]:
    w0, w = train_linear_regression(prepare_X(df_train, fill), y_train)
    y_pred = w0 + prepare_X(df_val, fill).dot(w)
    print(name, round(rmse(y_val, y_pred), 3))

With 0 2.205
With mean 2.202


## Question 4 - Best regularization

In [8]:
for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w0, w = train_linear_regression(prepare_X(df_train, 0), y_train, r=r)
    y_pred = w0 + prepare_X(df_val, 0).dot(w)
    print(r, round(rmse(y_val, y_pred), 4))

0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292


## Question 5 - RMSE standard deviation

In [9]:
scores = []
for seed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    df_train, df_val, df_test = split_data(df, seed)
    w0, w = train_linear_regression(prepare_X(df_train, 0), df_train.fuel_efficiency_mpg.values)
    y_pred = w0 + prepare_X(df_val, 0).dot(w)
    scores.append(rmse(df_val.fuel_efficiency_mpg.values, y_pred))

print(np.round(scores, 4))
round(np.std(scores), 3)

[2.2392 2.2021 2.1634 2.2044 2.2019 2.2458 2.2697 2.1908 2.2166 2.207 ]


np.float64(0.029)

## Question 6 - Evaluation on test

In [10]:
df_train, df_val, df_test = split_data(df, 9)
df_full_train = pd.concat([df_train, df_val]).reset_index(drop=True)

w0, w = train_linear_regression(prepare_X(df_full_train, 0), df_full_train.fuel_efficiency_mpg.values, r=0.001)
y_pred = w0 + prepare_X(df_test, 0).dot(w)
round(rmse(df_test.fuel_efficiency_mpg.values, y_pred), 3)

np.float64(2.236)

## Respostas

1. Missing values: `horsepower`
2. Median for horsepower: 254
3. Filling NAs: With mean
4. Best regularization: 0
5. RMSE standard deviation: 0.029
6. Evaluation on test: 2.236